# Conformal Verification Notebook

Compact evaluation flow: configure, prepare, run, summarize, and visualize.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from batch_generation import generate_random_batch
from plot_functions import plot_pb_trajectories
from evaluation_setup import EvaluationConfig, prepare_evaluation_context

In [ ]:
checkpoint_path = 'ren_standard_checkpoint.pt'
device = 'cpu'

horizon = 300
num_test_samples = 500
x0_std = 0.2
noise_std = 0.0

config = EvaluationConfig()

In [ ]:
ctx = prepare_evaluation_context(
    checkpoint_path=checkpoint_path,
    device=device,
    config=config
)

eval_system = ctx['eval_system']
metric = ctx['metric']
x_target = ctx['x_target']
obs_centers = ctx['obs_centers']
obs_sigmas = ctx['obs_sigmas']

print('Evaluation context is ready.')
print(f"REN dims: {ctx['dims']}")
print(f"Missing keys: {list(ctx['load_result'].missing_keys)}")
print(f"Unexpected keys: {list(ctx['load_result'].unexpected_keys)}")

In [ ]:
test_w = generate_random_batch(
    batch_size=num_test_samples,
    horizon=horizon,
    n_agents=config.n_agents,
    x0_std=x0_std,
    noise_std=noise_std,
    device=ctx['device']
)

pb_loop = eval_system['pb_loop']
pb_loop.eval()

with torch.no_grad():
    traj_x, traj_u, traj_w_hat = pb_loop.run(test_w)

print(f'Test set shape: {test_w.shape}')
print(f'Trajectory x shape: {traj_x.shape}')
print(f'Trajectory u shape: {traj_u.shape}')
print(f'Trajectory w_hat shape: {traj_w_hat.shape}')

In [ ]:
with torch.no_grad():
    costs = metric.compute_per_sample_cost(traj_x, traj_u).cpu().numpy()

mean_cost = np.mean(costs)
std_cost = np.std(costs)
min_cost = np.min(costs)
max_cost = np.max(costs)
q50 = np.quantile(costs, 0.50)
q95 = np.quantile(costs, 0.95)

print('=' * 60)
print('PERFORMANCE STATISTICS (Test Set)')
print('=' * 60)
print(f'Mean Cost:         {mean_cost:.4f}')
print(f'Std Dev:           {std_cost:.4f}')
print(f'Median (50%):      {q50:.4f}')
print(f'95th Percentile:   {q95:.4f}')
print(f'Minimum Cost:      {min_cost:.4f}')
print(f'Maximum Cost:      {max_cost:.4f}')
print(f'Test Set Size:     {num_test_samples}')
print('=' * 60)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(costs, bins=50, kde=True, stat='density', color='steelblue')
plt.axvline(mean_cost, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_cost:.3f}')
plt.axvline(q95, color='orange', linestyle='--', linewidth=2, label=f'95%: {q95:.3f}')
plt.xlabel('Cost')
plt.ylabel('Density')
plt.title('Cost Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([costs], labels=['Test Set'], patch_artist=True)
plt.ylabel('Cost')
plt.title('Cost Box Plot')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
plot_pb_trajectories(
    traj_x=traj_x,
    traj_u=traj_u,
    traj_w_hat=traj_w_hat,
    x_target=x_target,
    obs_centers=obs_centers,
    obs_sigma=obs_sigmas,
    dt=config.dt
)